# MethylLlama — Quickstart

This notebook loads the pretrained MethylLlama checkpoint from HuggingFace,
runs it on the 120-sample demo dataset, and shows age predictions.

**What you need**
- Python 3.10+, PyTorch, and the packages in `requirements.txt`
- GPU recommended but not required (CPU runs slowly on 21k CpGs)

**Architecture reminder**
```
Input: N CpG sites (β-values + site IDs)
  → MethylLlamaEmbeddings  (CpG-ID embed + ScaleAdaptEncoder)
  → 4 × LlamaLayer         (RMSNorm + MHA/RoPE + SwiGLU, Pre-LN)
  → CLS pooler_output      (Linear + Tanh)
  → age head               (256D → 1)
```
The pretraining task was WCED (Whole-Cell Expression Decoder): the CLS token
learns to reconstruct all CpG β-values from a 50 % random input subset,
forcing it to summarise the full methylation profile.

In [ ]:
# Install / upgrade if running for the first time
# !pip install -q huggingface_hub anndata scanpy matplotlib

In [ ]:
import sys, os
from pathlib import Path

# Make sure bmfm_methylation is importable (works when notebook is in tutorials/)
repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Download checkpoint from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="pretrain_llama_epoch98_val0.0059.ckpt",
)
print(f"Checkpoint at: {ckpt_path}")

## 2. Inspect checkpoint & build model config

In [ ]:
# Patch torch.load for PyTorch >= 2.6 (weights_only=True breaks Lightning ckpts)
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

ckpt = torch.load(ckpt_path, map_location="cpu")

# Determine vocab_size from the actual embedding table stored in the checkpoint
embed_weight = ckpt["state_dict"]["encoder.embeddings.cpg_sites_embeddings.weight"]
vocab_size   = embed_weight.shape[0]   # e.g. 49161 = 49156 CpGs + 5 special tokens
hidden_size  = embed_weight.shape[1]   # 256 for the 'small' architecture

print(f"Embedding table:  {vocab_size} × {hidden_size}")
print(f"Decoder vocab:    {ckpt['hyper_parameters']['vocab_size']} CpGs per batch (WCED subset_k)")
print(f"Hparams:          {ckpt['hyper_parameters']}")

In [ ]:
from bmfm_methylation.llama.model import MethylLlamaConfig
from bmfm_methylation.llama.wced_llama import WCEDLlamaModule

# Architecture: 'small' variant (256D × 4L × 4H), matches pretrain job 44450919
model_config = MethylLlamaConfig(
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=320,
    vocab_size=vocab_size,      # from checkpoint embedding table
    n_sin_basis=48,
    basis_scale=2.0,
)

decoder_vocab = ckpt["hyper_parameters"]["vocab_size"]
module = WCEDLlamaModule(
    model_config=model_config,
    vocab_size=decoder_vocab,
)
module.load_state_dict(ckpt["state_dict"])
module = module.to(DEVICE).eval()

encoder = module.encoder
n_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder loaded: {n_params/1e6:.1f}M parameters")

## 3. Load demo dataset

120 samples × 21,368 CpGs — stratified by age bin, diverse tissues.

In [ ]:
demo_path = Path("data/methylllama_demo_500samples.h5ad")
adata = ad.read_h5ad(demo_path)

print(f"Shape:    {adata.shape}")
print(f"Obs cols: {list(adata.obs.columns)}")
print(f"\nAge range: {float(adata.obs['age'].min()):.0f} – {float(adata.obs['age'].max()):.0f} yr")
print(f"Tissues (top 8):\n{adata.obs['tissue_type'].value_counts().head(8)}")

## 4. Load tokenizer & tokenize

MethylLlama expects dual-field input: `[B, 2, L]`
- Field 0: CpG-site token IDs (integer)
- Field 1: β-values (float, −2.0 marks the CLS position)

We download the pretrained 49k-CpG vocabulary from HuggingFace — the exact
tokenizer used during pretraining — so every CpG gets its correct token ID.
Demo CpGs not present in the 49k vocab automatically map to UNK.

In [ ]:
from huggingface_hub import hf_hub_download

# Download the pretrained 49k-CpG vocabulary from HuggingFace
vocab_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="tokenizer/cpg_sites/vocab.txt",
)

# Build a plain dict: token → id  (BertTokenizer ignores vocab_file in modern transformers)
cpg_vocab = {}
with open(vocab_path) as f:
    for idx, line in enumerate(f):
        cpg_vocab[line.strip()] = idx

cls_id = cpg_vocab["[CLS]"]
unk_id = cpg_vocab["[UNK]"]
cpg_names = list(adata.var_names)

demo_ids = [cpg_vocab.get(c, unk_id) for c in cpg_names]
n_known = sum(1 for i in demo_ids if i != unk_id)
print(f"Vocabulary: {len(cpg_vocab)} tokens  |  CLS={cls_id}  UNK={unk_id}")
print(f"Demo CpGs in pretrained vocab: {n_known}/{len(cpg_names)} ({100*n_known/len(cpg_names):.1f}%)")

In [ ]:
def tokenize_adata(adata, cpg_vocab, n_cpg=4000, seed=42):
    """
    Convert an h5ad into MethylLlama inputs.

    Selects n_cpg known sites at random (matches WCED pretraining input ratio).
    Returns:
        input_ids:      [N, 2, n_cpg+1]  (field 0 = cpg_ids, field 1 = beta_values)
        attention_mask: [N, n_cpg+1]
        ages:           [N] float tensor
    """
    rng = np.random.default_rng(seed)
    unk_id = cpg_vocab["[UNK]"]
    cls_id = cpg_vocab["[CLS]"]

    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = X.astype(np.float32)

    names = list(adata.var_names)
    all_ids = np.array([cpg_vocab.get(c, unk_id) for c in names])

    # Use only CpGs present in the pretrained vocab
    known_mask = all_ids != unk_id
    known_idx  = np.where(known_mask)[0]
    n_cpg = min(n_cpg, len(known_idx))

    sel_local = rng.choice(len(known_idx), size=n_cpg, replace=False)
    sel_idx   = known_idx[sel_local]
    sel_token_ids = all_ids[sel_idx].tolist()

    L = n_cpg + 1  # +1 for CLS

    all_input_ids, all_masks = [], []

    for i in range(adata.shape[0]):
        betas = X[i, sel_idx].copy()
        betas[np.isnan(betas)] = 0.5

        cpg_id_t  = torch.tensor([cls_id] + sel_token_ids, dtype=torch.float32)
        beta_val_t = torch.tensor([-2.0] + betas.tolist(), dtype=torch.float32)

        all_input_ids.append(torch.stack([cpg_id_t, beta_val_t]))   # [2, L]
        all_masks.append(torch.ones(L, dtype=torch.long))

    input_ids_t      = torch.stack(all_input_ids)   # [N, 2, L]
    attention_mask_t = torch.stack(all_masks)         # [N, L]
    ages_t = torch.tensor(adata.obs["age"].astype(float).values, dtype=torch.float32)

    return input_ids_t, attention_mask_t, ages_t


input_ids, attention_mask, ages = tokenize_adata(adata, cpg_vocab, n_cpg=4000)
print(f"input_ids:      {tuple(input_ids.shape)}")
print(f"attention_mask: {tuple(attention_mask.shape)}")
print(f"ages:           {tuple(ages.shape)}  min={ages.min():.0f}  max={ages.max():.0f}")

## 5. Extract CLS embeddings

`pooler_output` = Linear(CLS hidden state) + Tanh → global methylation summary

In [ ]:
BATCH_SIZE = 16
all_cls = []

with torch.no_grad():
    for start in range(0, len(input_ids), BATCH_SIZE):
        ids_b  = input_ids[start:start+BATCH_SIZE].to(DEVICE)
        mask_b = attention_mask[start:start+BATCH_SIZE].to(DEVICE)

        out = encoder(input_ids=ids_b, attention_mask=mask_b)
        all_cls.append(out.pooler_output.cpu())

cls_embeddings = torch.cat(all_cls, dim=0).numpy()  # [120, 256]
print(f"CLS embeddings: {cls_embeddings.shape}")
print(f"Norm (mean ± std): {np.linalg.norm(cls_embeddings, axis=1).mean():.3f} "
      f"± {np.linalg.norm(cls_embeddings, axis=1).std():.3f}")

## 6. Age signal in the CLS embedding space

The WCED pretraining used `age_weight=0.0` — the auxiliary age head contributed zero
to the loss and its weights are random initialisation. **Do not use the pretrain head
for age predictions.**

Instead, we measure how much age information is *implicitly* encoded in the CLS vectors
by computing the Pearson correlation between true age and the **first PCA component** of
the embedding matrix.

> For full age prediction (R²=0.905, MedAE=3.65 yr) a fine-tuned checkpoint is needed.
> See `age_prediction.ipynb` for the paper results.

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

true_ages = ages.numpy()

pca = PCA(n_components=10, random_state=42)
pcs = pca.fit_transform(cls_embeddings)

print("PC correlations with true age:")
for i in range(10):
    r, p = pearsonr(true_ages, pcs[:, i])
    print(f"  PC{i+1}: r = {r:+.3f}  (p={p:.2e})")

best_pc = int(np.argmax([abs(pearsonr(true_ages, pcs[:, i])[0]) for i in range(10)]))
r_best, _ = pearsonr(true_ages, pcs[:, best_pc])

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(true_ages, pcs[:, best_pc],
                c=true_ages, cmap="plasma", s=45, alpha=0.80, edgecolors="none")
cb = plt.colorbar(sc, ax=ax, label="Age (yr)", pad=0.02)
cb.ax.tick_params(labelsize=11)
ax.set_xlabel("Chronological age (yr)", fontsize=13)
ax.set_ylabel(f"PC{best_pc+1} of CLS embedding", fontsize=13)
ax.set_title(
    f"Age signal in MethylLlama CLS space\n"
    f"(pretrained only, no fine-tuning)   r = {r_best:.3f}",
    fontsize=13,
)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig("cls_age_pca.png", dpi=300)
plt.show()
print(f"\nSaved: cls_age_pca.png")
print("For full age prediction: see age_prediction.ipynb  (fine-tuned R²=0.905)")

## Next steps

- **Embedding analysis**: see `embedding_analysis.ipynb` for UMAP visualisation of the CLS embeddings coloured by age and tissue type
- **Fine-tuning**: see `age_prediction.ipynb` for the full fine-tuning pipeline and paper results
- **Cluster training**: scripts in `scripts/llama/` and `scripts/wced/` for SLURM submission